# NASA Turbofan Motorları — Kalan Faydalı Ömür (RUL) Tahmini — FD001

Bu çalışmada NASA'nın **C-MAPSS** veri setini kullanarak uçak motorlarının **kalan faydalı ömrünü (RUL — Remaining Useful Life)** tahmin edeceğiz.

Her motor ilk başta sağlıklı çalışıyor, zamanla bir arıza gelişiyor ve sonunda bozuluyor. Amacımız: motorun sensör ölçümlerine bakarak **"bu motor bozulmadan önce daha kaç çevrim (uçuş) çalışabilir?"** sorusunu cevaplamak.

**FD001 veri setinin özellikleri:**
- Eğitim (train) motor sayısı: **100**
- Test motor sayısı: **100**
- Çalışma koşulu sayısı: **1 (deniz seviyesi)**
- Arıza tipi sayısı: **1 (HPC — yüksek basınçlı kompresör bozulması)**

## Kaggle'da veriyi ekle

Bu defter Kaggle için uyarlandı. Sağ paneldeki **Add Input** ile NASA C-MAPSS veri setini ekle (ör. `nasa-cmaps` adlı topluluk veri seti; içinde `train_FDxxx.txt`, `test_FDxxx.txt`, `RUL_FDxxx.txt` dosyaları bulunmalı). Aşağıdaki hücre bu dosyaları `/kaggle/input` içinde bulup çalışma klasörüne kopyalar; defterin geri kalanı değişmeden çalışır.

*Kaggle version note: add the NASA C-MAPSS dataset through "Add Input"; the next cell finds and copies the files automatically.*

In [ ]:
import os, shutil
DS = "FD001"
bulunan = {}
for kok, _, dosyalar in os.walk("/kaggle/input"):
    for f in dosyalar:
        if f in (f"train_{DS}.txt", f"test_{DS}.txt", f"RUL_{DS}.txt"):
            bulunan[f] = os.path.join(kok, f)
for f, yol in bulunan.items():
    shutil.copy(yol, f)
print("Kopyalanan dosyalar:", sorted(bulunan))
assert len(bulunan) == 3, "3 dosya bulunamadi: Add Input ile C-MAPSS veri setini ekle."

Aşağıdaki hücreyi çalıştırarak kullanacağımız kütüphaneleri yükleyin.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score

## 1. TXT dosyalarını CSV'ye çevirme

Dosyalarımız `.txt` formatında: sütunlar **boşlukla** ayrılmış ve **sütun başlığı yok**. Önce bunları düzgün sütun adlarına sahip birer tabloya (DataFrame) çevirip **CSV** (virgülle ayrılmış değerler) olarak kaydedeceğiz.

`readme.txt` dosyasına göre 26 sütun var: motor numarası, çevrim, 3 çalışma ayarı ve 21 sensör. Aşağıdaki hücreyi çalıştırarak sütun adlarını tanımlayın.

In [ ]:
column_names = ["unit", "cycle", "setting_1", "setting_2", "setting_3",
                "sensor_1", "sensor_2", "sensor_3", "sensor_4", "sensor_5",
                "sensor_6", "sensor_7", "sensor_8", "sensor_9", "sensor_10",
                "sensor_11", "sensor_12", "sensor_13", "sensor_14", "sensor_15",
                "sensor_16", "sensor_17", "sensor_18", "sensor_19", "sensor_20",
                "sensor_21"]

`train_FD001.txt` dosyasını `df_train` adlı bir DataFrame'e yükleyin.

<details>
    <summary> İpucu 💡 </summary>

`pd.read_csv()` fonksiyonunu üç ek argümanla kullanabilirsiniz:
- `sep=r"\s+"` → sütunlar bir veya daha fazla boşlukla ayrılmış
- `header=None` → dosyada başlık satırı yok
- `names=column_names` → sütun adlarını biz veriyoruz

</details>

In [ ]:
# your code here

Aynı şekilde `test_FD001.txt` dosyasını `df_test` içine yükleyin.

`RUL_FD001.txt` dosyasında ise tek bir sütun var. Onu `df_rul` içine, sütun adı `"RUL"` olacak şekilde yükleyin.

In [ ]:
# your code here

Üç tabloyu da `.to_csv()` ile CSV olarak kaydedin. Satır numaralarının dosyaya yazılmaması için `index=False` kullanın.

In [ ]:
# your code here

Bundan sonra CSV dosyalarıyla çalışacağız. Üç CSV dosyasını tekrar `df_train`, `df_test` ve `df_rul` içine okuyun.

In [ ]:
# your code here

## 2. Keşif (Exploratory Analysis)

`df_train` tablosunun ilk 5 satırına bakın.

In [ ]:
# your code here

Bir satır neyi temsil eder? Hangi bilgileri bulabilirsiniz?

> **Cevabınızı buraya yazın (çift tıklayın)**

### Sensör sözlüğü
Sütunlarda `sensor_1`, `sensor_2`... yazıyor ama bunların **gerçekte ne ölçtüğünü** bilmeden yorum yapmak zor. NASA'nın makalesindeki sensör açıklamalarını bir **sözlük (dictionary)** içine koyduk. Sözlük, bir anahtara (`"sensor_1"`) karşılık bir değer (`"(Fan inlet temperature) (◦R)"`) saklar.

Aşağıdaki hücreyi çalıştırın.

In [ ]:
Sensor_dictionary = {}
dict_list = ["(Fan inlet temperature) (◦R)",
             "(LPC outlet temperature) (◦R)",
             "(HPC outlet temperature) (◦R)",
             "(LPT outlet temperature) (◦R)",
             "(Fan inlet Pressure) (psia)",
             "(bypass-duct pressure) (psia)",
             "(HPC outlet pressure) (psia)",
             "(Physical fan speed) (rpm)",
             "(Physical core speed) (rpm)",
             "(Engine pressure ratio(P50/P2)",
             "(HPC outlet Static pressure) (psia)",
             "(Ratio of fuel flow to Ps30) (pps/psia)",
             "(Corrected fan speed) (rpm)",
             "(Corrected core speed) (rpm)",
             "(Bypass Ratio) ",
             "(Burner fuel-air ratio)",
             "(Bleed Enthalpy)",
             "(Required fan speed)",
             "(Required fan conversion speed)",
             "(High-pressure turbines Cool air flow)",
             "(Low-pressure turbines Cool air flow)"]

i = 1
for x in dict_list:
    Sensor_dictionary["sensor_" + str(i)] = x
    i += 1

pd.Series(Sensor_dictionary, name="Ne ölçüyor?").to_frame()

Kısaltmalar: **Fan** = ön pervane, **LPC / HPC** = alçak / yüksek basınçlı kompresör, **LPT** = alçak basınçlı türbin, **◦R** = Rankine (sıcaklık birimi), **psia** = basınç birimi, **rpm** = dakikadaki devir. Yani sensörler motorun farklı bölümlerindeki **sıcaklık, basınç ve devir** değerlerini ölçüyor.

Tabloların kaç satır ve kaç sütundan oluştuğuna bakın. Train ve test verisinde kaç farklı motor var?

<details>
    <summary> İpucu 💡 </summary>

`.shape` satır ve sütun sayısını verir. `.nunique()` bir sütundaki farklı değerlerin sayısını verir.

</details>

In [ ]:
# your code here

`.describe()` ile sayısal sütunların özet istatistiklerine bakın.

In [ ]:
# your code here

`describe()` bize her sütun için şunları verir:
- `count`: dolu değer sayısı
- `mean`: ortalama
- `std`: standart sapma → değerler ortalamadan ne kadar uzaklaşıyor? **0'a yakınsa sütun neredeyse hiç değişmiyor demektir.**
- `min` / `max`: en küçük / en büyük değer
- `25%`, `50%`, `75%`: çeyrekler (`50%` = medyan)

Hangi sütunların `std` değeri 0 veya 0'a çok yakın? Bu sütunlar model için faydalı olur mu?

> **Cevabınızı buraya yazın (çift tıklayın)**

Tabloda boş (`NaN`) değer var mı? Her sütun için kontrol edin.

<details>
    <summary> İpucu 💡 </summary>

`.isnull()` her hücre için boş mu değil mi (True/False) döndürür. Arkasına `.sum()` eklerseniz her sütundaki boş değer sayısını verir.

</details>

In [ ]:
# your code here

Değeri tam olarak **0** olan hücreleri sayın. Hangi sütunlarda 0 değeri var?

In [ ]:
# your code here

Boş ve sıfır değer kontrolünün sonucu nedir? Sıfır değerleri bir sorun mu?

> **Cevabınızı buraya yazın (çift tıklayın)**

Şimdi **medyan** ve **mod** değerlerine bakalım:
- **Medyan:** Değerleri küçükten büyüğe sıraladığımızda tam ortada kalan değer. Uç değerlerden (çok büyük/küçük değerlerden) etkilenmez.
- **Mod:** En sık tekrar eden değer.

`sensor_11` sütununun ortalamasını (`mean`), medyanını (`median`) ve modunu (`mode`) hesaplayın.

In [ ]:
# your code here

Ortalama ile medyan birbirine yakın mı? Bu bize ne söyler?

> **Cevabınızı buraya yazın (çift tıklayın)**

`sensor_1` sütununda her değerin kaç kez tekrar ettiğini sayın. Mod (en sık görülen değer) kaç kez tekrar ediyor?

<details>
    <summary> İpucu 💡 </summary>

`.value_counts()` her değerin kaç kez tekrar ettiğini sayar.

</details>

In [ ]:
# your code here

> **Cevabınızı buraya yazın (çift tıklayın)**

Her motorun arızalanana kadar kaç çevrim çalıştığını hesaplayın. Sonra bu değerlerin özet istatistiğine bakın.

<details>
    <summary> İpucu 💡 </summary>

Her motor için en büyük `cycle` değeri, o motorun ömrüdür: `groupby("unit")["cycle"].max()`

</details>

In [ ]:
# your code here

Motor ömürlerinin dağılımını bir histogram ile çizin.

In [ ]:
# your code here

En kısa ve en uzun ömürlü motor kaç çevrim çalışmış? Bütün motorlar aynı sürede mi bozuluyor?

> **Cevabınızı buraya yazın (çift tıklayın)**

Aşağıdaki kodu çalıştırarak **1 numaralı motorun** `sensor_11` değerinin zaman içinde nasıl değiştiğini görün.

In [ ]:
engine_1 = df_train[df_train["unit"] == 1]

plt.plot(engine_1["cycle"], engine_1["sensor_11"])
plt.xlabel("Çevrim")
plt.ylabel("sensor_11")
plt.title("1 numaralı motorun sensor_11 değeri")
plt.show()

Grafikte ne görüyorsunuz?

> **Cevabınızı buraya yazın (çift tıklayın)**

## 3. Temizlik: Bilgi vermeyen sütunları çıkarma

Std değeri 0.01'den küçük olan sütunları bulun ve listesini `constant_cols` değişkenine kaydedin.

<details>
    <summary> İpucu 💡 </summary>

```python
stds = df_train.std()
constant_cols = stds[stds < 0.01].index
```
İlk satır her sütunun std'sini hesaplar, ikinci satır std'si 0.01'den küçük olanların adlarını seçer.

</details>

In [ ]:
# your code here

Sözlükten bakarak bu sütunların ne ölçtüğünü görün. (`.get()`, sözlükte karşılığı olmayan anahtarlar için "çalışma ayarı" yazar; `setting` sütunları sensör değildir.)

In [ ]:
for c in constant_cols:
    print(c, "→", Sensor_dictionary.get(c, "çalışma ayarı"))

Bu sütunları hem `df_train` hem de `df_test` tablosundan silin. (Modelin test verisinde de aynı sütunları görmesi gerekir!)

In [ ]:
# your code here

## 4. Hedef değişkeni oluşturma: RUL

Modelimizin tahmin edeceği değer **RUL**: motorun bozulmadan önce **kalan** çevrim sayısı. Train verisinde bu sütun hazır değil, biz hesaplayacağız.

Train verisinde her motor **arızalanana kadar** çalışıyor. Yani bir motorun son çevrimi, arıza anı. Buna göre:

**RUL = motorun son çevrimi − şu anki çevrim**

Örnek: Motor 192 çevrimde bozulduysa, 100. çevrimde RUL = 192 − 100 = 92.

Önce her motorun son çevrimini bulun ve `max_cycles` adlı bir tabloya kaydedin. Sütun adları `unit` ve `max_cycle` olsun.

In [ ]:
# your code here

`max_cycles` tablosunu `unit` sütunu üzerinden `df_train` ile birleştirin (merge). Ardından `RUL` sütununu hesaplayın ve artık gerekmeyen `max_cycle` sütununu silin.

<details>
    <summary> İpucu 💡 </summary>

```python
df_train = df_train.merge(max_cycles, on="unit")
```
Ardından: `df_train["RUL"] = df_train["max_cycle"] - df_train["cycle"]`

</details>

In [ ]:
# your code here

Kontrol: 1 numaralı motorun **son 5 satırına** bakın. Son satırda RUL kaç olmalı?

In [ ]:
# your code here

Her sensörün RUL ile **korelasyonunu** hesaplayın ve küçükten büyüğe sıralayın.

<details>
    <summary> İpucu 💡 </summary>

`df_train.corr()` tüm sütunlar arasındaki korelasyonu verir. Sadece RUL sütununu almak için: `df_train.corr()["RUL"]`

Korelasyon -1 ile +1 arasındadır:
- **+1'e yakın:** sensör arttıkça RUL da artıyor
- **-1'e yakın:** sensör arttıkça RUL azalıyor (motor bozulmaya yaklaşıyor)
- **0'a yakın:** belirgin bir ilişki yok

</details>

In [ ]:
# your code here

Aşağıdaki kodu çalıştırarak korelasyonları bir ısı haritası (heatmap) olarak görün.

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df_train.corr(), cmap="coolwarm", center=0)
plt.title("Korelasyon ısı haritası")
plt.show()

Aşağıdaki kodu çalıştırarak RUL ile en güçlü ilişkiye sahip 5 sensörü ve **ne ölçtüklerini** görün.

In [ ]:
corr = df_train.corr()["RUL"]
corr = corr[corr.index.str.startswith("sensor")]
top5 = corr.abs().sort_values(ascending=False).head(5).index

for s in top5:
    print(s, "| korelasyon:", round(corr[s], 2), "|", Sensor_dictionary[s])

RUL ile en güçlü ilişkiye sahip sensörler hangileri? Sözlüğe bakarak ne ölçtüklerini de yazın.

> **Cevabınızı buraya yazın (çift tıklayın)**

## 5. Test verisini hazırlama

Test verisi train'den farklı: test motorları **arızadan bir süre önce** kesiliyor. Yani her test motorunun **son satırındaki** durumu biliyoruz ve `RUL_FD001.txt` dosyası bize **o son satırdan sonra** motorun kaç çevrim daha çalıştığını (gerçek RUL'u) söylüyor.

Bu yüzden test verisinde her motorun **sadece son satırını** alacağız. Her motorun son satırını `df_test_last` içine kaydedin.

<details>
    <summary> İpucu 💡 </summary>

`df_test.groupby("unit").last().reset_index()` her motor için son satırı alır.

</details>

In [ ]:
# your code here

`df_rul` tablosundaki gerçek RUL değerlerini `df_test_last` tablosuna `RUL` adlı yeni bir sütun olarak ekleyin. (İki tabloda da motorlar 1'den başlayarak aynı sırada.)

In [ ]:
# your code here

Son olarak modelde kullanacağımız **özellikleri (features)** belirleyelim: `unit`, `cycle` ve `RUL` dışındaki tüm sütunlar.

> `unit` sadece bir motor numarası, motorun sağlığı hakkında bilgi vermez. `cycle` testte motorun ömrünün ortasında da kesilebilir, yanıltıcı olur. `RUL` ise tahmin etmek istediğimiz şeyin kendisi!

In [ ]:
# your code here

### Özellikleri ölçekleme: MinMaxScaler

Sensörlerin değer aralıkları çok farklı: biri 8-9 civarında, diğeri 9000 civarında. **MinMaxScaler** her sütunu **0 ile 1 arasına** sıkıştırır:

**yeni değer = (değer − sütunun min'i) / (sütunun max'ı − min'i)**

En küçük değer 0, en büyük değer 1 olur. Lojistik Regresyon, SVR ve KNN gibi modeller büyük sayılı sütunu gereğinden fazla önemser, ölçekleme bunu eşitler. (Düz Lineer Regresyonun tahminleri ölçeklemeden etkilenmez; ölçekleme yalnızca katsayıları karşılaştırmayı kolaylaştırır.) **Random Forest'in ölçeklemeye ihtiyacı yoktur** (ağaçlar değerleri büyüklüğe göre değil eşiklere göre böler), bu yüzden aşağıdaki ölçekli veriyi bu çalışmadaki modellerde kullanmayacağız; amaç ölçeklemenin ne yaptığını görmek.

> **Önemli:** `fit_transform` sadece **train** verisinde kullanılır (min ve max train'den öğrenilir). Test verisine aynı kural `transform` ile uygulanır. Test verisinde `fit_transform` kullanmak, test verisinden bilgi sızdırmak demektir.

Aşağıdaki kodu çalıştırın.

In [ ]:
minmax_scaler = MinMaxScaler()
X_train_s = pd.DataFrame(minmax_scaler.fit_transform(df_train[features]), columns=features)
X_test_s = pd.DataFrame(minmax_scaler.transform(df_test_last[features]), columns=features)

print("Train min:", round(X_train_s.min().min(), 2), "| Train max:", round(X_train_s.max().max(), 2))

Ölçeklenmiş train verisinin her özelliği için histogram çizin. Dağılımların şekli değişmez, sadece değer aralığı 0-1 olur.

In [ ]:
X_train_s.hist(bins=50, figsize=(16, 12))
plt.tight_layout()
plt.show()

## 6. Model 1: Lineer Regresyon

Tahmin etmek istediğimiz RUL **sürekli bir sayı** (örneğin 87 çevrim). Sayı tahmin eden modellere **regresyon** modeli denir. En basit regresyon modeli olan **Lineer Regresyon** ile başlıyoruz.

Modeli eğitmek için girdileri (`X`) ve tahmin edilecek değeri (`y`) ayırın:
- `X_train`: `df_train` içindeki `features` sütunları
- `y_train`: `df_train` içindeki `RUL` sütunu
- `X_test` ve `y_test`: aynısı, `df_test_last` için

In [ ]:
# your code here

Bir `LinearRegression` modeli oluşturun, `X_train` ve `y_train` ile eğitin (`fit`), ardından `X_test` için tahmin yapın (`predict`) ve tahminleri `y_pred_lr` içine kaydedin.

In [ ]:
# your code here

Aşağıdaki kodu çalıştırarak ilk 10 test motoru için gerçek ve tahmin edilen değerleri yan yana görün.

In [ ]:
comparison = pd.DataFrame({"Gerçek RUL": y_test, "Tahmin": y_pred_lr.round(0)})
comparison.head(10)

Şimdi modelin ne kadar iyi olduğunu ölçelim. İki metrik kullanacağız:
- **MAE (Ortalama Mutlak Hata):** Tahminlerimiz ortalamada kaç çevrim yanılıyor?
- **RMSE (Kök Ortalama Kare Hata):** Hataların karesini alıp ortalar, sonra karekök alır. Büyük hataları daha çok cezalandırır.

İkisi de **ne kadar küçükse o kadar iyi**. RMSE'yi `np.sqrt(mean_squared_error(...))`, MAE'yi `mean_absolute_error(...)` ile hesaplayın ve `rmse_lr`, `mae_lr` değişkenlerine kaydedin.

In [ ]:
# your code here

Bu sonucu nasıl yorumlarsınız? Bir uçak bakım mühendisi için yeterli mi?

> **Cevabınızı buraya yazın (çift tıklayın)**

## 7. İyileştirme: RUL'u 125'te sınırlama

Bir sorun var: Motor ilk çevrimlerde **tamamen sağlıklı**. Sensörler "bu motorun 300 çevrim mi yoksa 200 çevrim mi ömrü kaldı" farkını gösteremez; çünkü ikisinde de motor sağlıklı ve sensörler aynı.

Ama biz modelden, 1. çevrimdeki bir motor için RUL=300 tahmin etmesini istiyoruz. Bu, modelin öğrenmesi imkânsız bir şey!

**Çözüm:** RUL değerini 125'te sınırlayalım (clipping). "125'ten fazla ömrü kalan motor = sağlıklı motor" diyoruz. Bu yöntem bu veri setiyle yapılan bilimsel çalışmalarda çok yaygın kullanılır.

`df_train` tablosuna `RUL_clip` adında yeni bir sütun ekleyin: RUL değeri 125'ten büyükse 125 olsun, değilse aynı kalsın.

<details>
    <summary> İpucu 💡 </summary>

`.clip(upper=125)` 125'ten büyük değerleri 125'e eşitler.

</details>

In [ ]:
# your code here

Aşağıdaki kodu çalıştırarak 1 numaralı motor için sınırlanmış ve sınırlanmamış RUL'u karşılaştırın.

In [ ]:
engine_1 = df_train[df_train["unit"] == 1]

plt.plot(engine_1["cycle"], engine_1["RUL"], label="RUL (sınırsız)")
plt.plot(engine_1["cycle"], engine_1["RUL_clip"], label="RUL (125'te sınırlı)")
plt.xlabel("Çevrim")
plt.ylabel("RUL")
plt.legend()
plt.title("1 numaralı motor: RUL sınırlama")
plt.show()

Modelimiz artık en fazla 125 tahmin etmeyi öğrenecek. Adil bir karşılaştırma için test verisindeki gerçek RUL değerlerini de 125'te sınırlamalıyız.

`y_train_clip` ve `y_test_clip` değişkenlerini oluşturun.

In [ ]:
# your code here

Yeni bir Lineer Regresyon modelini `X_train` ve `y_train_clip` ile eğitin. Tahminleri `y_pred_lr_clip` içine kaydedin, ardından `y_test_clip` ile RMSE ve MAE hesaplayın (`rmse_lr_clip`, `mae_lr_clip`).

In [ ]:
# your code here

RMSE nasıl değişti? Sizce neden?

> **Cevabınızı buraya yazın (çift tıklayın)**

## 8. Model 2: Random Forest

Lineer Regresyon, sensörler ile RUL arasında **düz bir çizgi** gibi bir ilişki varsayar. Gerçekte ilişki daha karmaşık olabilir: motor uzun süre sağlıklı kalır, sonra hızla bozulur.

**Random Forest**, birçok **karar ağacının** tahminlerinin ortalamasını alan bir modeldir. Her ağaç "sensor_11 > 47.5 mi? Evet ise..." gibi sorularla tahmin yapar. Bu sayede doğrusal olmayan ilişkileri de yakalayabilir.

Bir `RandomForestRegressor` modeli oluşturun:
- `n_estimators=100` → 100 ağaç
- `max_depth=10` → her ağaç en fazla 10 soru derinliğinde (aşırı ezberlemeyi önler)
- `random_state=42` → her çalıştırmada aynı sonuç
- `n_jobs=-1` → bilgisayarın tüm işlemci çekirdeklerini kullan (daha hızlı)

Modeli `X_train` ve `y_train_clip` ile eğitin, `X_test` için tahmin yapın (`y_pred_rf`), RMSE ve MAE hesaplayın (`rmse_rf`, `mae_rf`).

In [ ]:
# your code here

### Ezberleme kontrolü

Bir model eğitim verisini **ezberleyip** test verisinde kötü çıkabilir. Bunu anlamak için modelin **eğitim** verisindeki hatasına da bakarız.

`X_train` için tahmin yapın (`y_pred_rf_train`), `y_train_clip` ile RMSE hesaplayıp `rmse_rf_train` içine kaydedin. Sonra eğitim ve test RMSE'sini yan yana yazdırın.

In [ ]:
# your code here

Eğitim ve test hatası arasında büyük bir fark var mı? Bu ne anlama gelir?

> **Cevabınızı buraya yazın (çift tıklayın)**

Random Forest bize hangi sensörün tahmin için ne kadar önemli olduğunu da söyler. Aşağıdaki kodu çalıştırın.

In [ ]:
importance = pd.Series(rf_model.feature_importances_, index=features)
importance.sort_values().plot(kind="barh", figsize=(8, 6))
plt.title("Random Forest: Özellik önemi")
plt.xlabel("Önem")
plt.show()

Aşağıdaki kodu çalıştırarak en önemli 3 özelliği ve ne ölçtüklerini görün.

In [ ]:
top3 = importance.sort_values(ascending=False).head(3)

for s in top3.index:
    print(s, "| önem:", round(top3[s], 2), "|", Sensor_dictionary.get(s, "çalışma ayarı"))

En önemli 3 sensör hangisi? 4. bölümdeki korelasyon sonuçlarıyla uyumlu mu?

> **Cevabınızı buraya yazın (çift tıklayın)**

## 9. Sonuçları karşılaştırma

Şimdi üçüncü bir metrik ekleyelim: **R² (belirlilik katsayısı)**. Modelin RUL'daki değişimin ne kadarını açıkladığını gösterir:
- **1'e yakın** → model çok iyi
- **0 civarı** → model, "herkese ortalamayı söyle" yönteminden farksız
- **Negatif** → model ortalamayı söylemekten bile kötü

RMSE ve MAE hatayı **çevrim cinsinden** verir, R² ise **genel uyumu** gösterir; ikisi birbirini tamamlar. `r2_score(gerçek, tahmin)` ile üç modelin R² değerlerini hesaplayın (`r2_lr`, `r2_lr_clip`, `r2_rf`). RMSE ve MAE'de olduğu gibi, sınırlanmış RUL ile eğitilen modeller `y_test_clip` ile karşılaştırılır.

In [ ]:
# your code here

Üç modelin RMSE, MAE ve R² değerlerini bir tabloda toplayın.

In [ ]:
# your code here

Aşağıdaki kodu çalıştırarak tahminleri gerçek değerlerle karşılaştırın. Her test motoru için **mavi çubuk gerçek RUL**, üstüne çizilen **ince kırmızı çubuk tahmin edilen RUL**'dur.

- Kırmızı çubuk mavinin **üstüne taşıyorsa**: model motorun ömrünü **olduğundan uzun** tahmin etmiş (bakım açısından tehlikeli yön).
- Kırmızı çubuk mavinin **içinde kalıyorsa**: model ömrü **olduğundan kısa** tahmin etmiş (temkinli hata).
- Çubuklar ne kadar yakınsa tahmin o kadar iyi. RUL 125'te sınırlandığı için hiçbir çubuk 125'i geçmez.

In [ ]:
def plot_predActual(y_true, y_pred, title):
    x = np.arange(len(y_true))
    plt.figure(figsize=(20, 7))
    plt.bar(x, y_true, width=0.6, color="blue", label="Gerçek RUL")
    plt.bar(x, y_pred, width=0.3, color="red", alpha=0.7, label="Tahmin edilen RUL")
    plt.xlabel("Test motoru")
    plt.ylabel("RUL")
    plt.title(title)
    plt.legend()
    plt.show()

plot_predActual(y_test_clip, y_pred_lr_clip, "Lineer Regresyon: Gerçek ve tahmin")
plot_predActual(y_test_clip, y_pred_rf, "Random Forest: Gerçek ve tahmin")

### Tahmin yönü: ömrü fazla mı, az mı tahmin ediyoruz?

Çubuk grafikte kırmızı çubuğun maviyi aşması, modelin ömrü **olduğundan uzun** tahmin etmesidir; bu bakım için **tehlikeli yöndür**. Bunu sayıyla görelim.

Random Forest için ömrü olduğundan uzun tahmin edilen motor sayısını `over` içine, kısa tahmin edilenleri `under` içine kaydedin.

<details>
    <summary> İpucu 💡 </summary>

`(y_pred_rf > y_test_clip).sum()` tahminin gerçekten büyük olduğu motorları sayar.

</details>

In [ ]:
# your code here

Model sistematik olarak iyimser mi (ömrü fazla tahmin ediyor) yoksa dengeli mi? Bakım açısından bu neden önemli?

> **Cevabınızı buraya yazın (çift tıklayın)**

### NASA skoru: geç tahmin daha pahalı

RMSE, fazla ve az tahmini **aynı ağırlıkta** cezalandırır. NASA'nın bu yarışma için tanımladığı **skor** ise ömrü **fazla tahmin etmeyi (geç bakım) daha ağır** cezalandırır. Hata `d = tahmin − gerçek` olsun:
- `d < 0` (az tahmin): ceza = `e^(−d/13) − 1`
- `d ≥ 0` (fazla tahmin): ceza = `e^(d/10) − 1`

Örneğin 10 çevrim fazla tahmin yaklaşık 1.72, 10 çevrim az tahmin yaklaşık 1.16 puan ceza getirir. Skor **ne kadar küçükse o kadar iyidir**.

`nasa_score(y_true, y_pred)` adında bir fonksiyon yazın (her motorun cezasını toplasın). Sonra Lineer Regresyon (RUL ≤ 125) ve Random Forest için skoru hesaplayıp `score_lr_clip` ve `score_rf` içine kaydedin.

<details>
    <summary> İpucu 💡 </summary>

`d = np.asarray(y_pred) - np.asarray(y_true)` ile hatayı bulun. `np.where(d < 0, ceza_az, ceza_fazla)` her motor için doğru formülü seçer, `.sum()` toplar.

</details>

In [ ]:
# your code here

> **Not:** Resmi NASA skoru RUL sınırlanmadan hesaplanır. Biz sınırlı RUL kullandığımız için sayılarımız literatürdeki skorlarla **birebir kıyaslanamaz**; burada yalnızca kendi modellerimizi birbirleriyle karşılaştırıyoruz.

NASA skoruna göre hangi model daha iyi? Sıralama RMSE ile aynı mı? Bu skor güvenlik açısından neden anlamlı?

> **Cevabınızı buraya yazın (çift tıklayın)**

Hangi model en iyi? Raporunuz için 2-3 cümlelik bir sonuç yazın.

> **Cevabınızı buraya yazın (çift tıklayın)**

## 10. Bonus: Sınıflandırma — "Motor 30 çevrim içinde arızalanacak mı?"

Bakım ekibi için bazen kesin sayıdan çok basit bir soru önemlidir: **"Bu motoru şimdi bakıma almalı mıyım?"**

Soruyu **evet/hayır** şekline çevirelim:
- RUL ≤ 30 ise → **1** (arızaya yakın, bakıma al)
- RUL > 30 ise → **0** (sağlam)

Cevap bir sayı değil bir **sınıf** olduğu için bu bir **sınıflandırma** problemi.

`df_train` ve `df_test_last` tablolarına `fail_30` adında bu etiketi içeren bir sütun ekleyin.

<details>
    <summary> İpucu 💡 </summary>

`(df_train["RUL"] <= 30)` True/False döndürür. `.astype(int)` ile True → 1, False → 0 olur.

</details>

In [ ]:
# your code here

Train verisinde sınıfların oranı nedir? `.value_counts()` fonksiyonunu `normalize=True` argümanıyla kullanın.

In [ ]:
# your code here

Sınıflar dengeli mi? Bu neden önemli olabilir?

> **Cevabınızı buraya yazın (çift tıklayın)**

**Lojistik Regresyon** ile sınıflandırma yapacağız. Bu model, sensör değerlerinin **farklı ölçeklerde** olmasından etkilenir (bir sensör 0-1 arası, diğeri 9000 civarında). Bu yüzden önce tüm özellikleri aynı ölçeğe getiriyoruz (**StandardScaler**: her sütunun ortalaması 0, std'si 1 olur).

Aşağıdaki kodu çalıştırın. Dikkat: `fit_transform` sadece train'de kullanılır; test verisine train'den öğrenilen ölçek `transform` ile uygulanır.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

`y_train_class` ve `y_test_class` değişkenlerini oluşturun. Bir `LogisticRegression(max_iter=1000)` modeli eğitin ve test tahminlerini `y_pred_class` içine kaydedin.

In [ ]:
# your code here

### Confusion Matrix (Karışıklık Matrisi)

Confusion matrix, modelin doğru ve yanlış tahminlerini 4 kutuda gösterir:

|  | Tahmin: 0 (sağlam) | Tahmin: 1 (arızaya yakın) |
|---|---|---|
| **Gerçek: 0 (sağlam)** | **TN** (True Negative) — doğru, sağlam dedi | **FP** (False Positive) — yanlış alarm |
| **Gerçek: 1 (arızaya yakın)** | **FN** (False Negative) — arızayı kaçırdı! | **TP** (True Positive) — doğru, arızayı yakaladı |

`confusion_matrix(y_test_class, y_pred_class)` ile matrisi hesaplayıp `cm` içine kaydedin ve yazdırın.

In [ ]:
# your code here

Aşağıdaki kodu çalıştırarak matrisi görselleştirin.

In [ ]:
ConfusionMatrixDisplay(cm, display_labels=["Sağlam (0)", "Arızaya yakın (1)"]).plot(cmap="Blues")
plt.title("Lojistik Regresyon: Confusion Matrix")
plt.show()

Matristeki 4 değeri ayrı değişkenlere alın. `cm.ravel()` matrisi düz bir listeye çevirir: sırası **TN, FP, FN, TP**.

In [ ]:
# your code here

Şimdi üç metrik hesaplayın:
- **Accuracy (Doğruluk)** = (TP + TN) / toplam → Tüm tahminlerin yüzde kaçı doğru?
- **Precision (Kesinlik)** = TP / (TP + FP) → "Arızaya yakın" dediğimiz motorların yüzde kaçı gerçekten arızaya yakın?
- **Recall (Duyarlılık)** = TP / (TP + FN) → Gerçekten arızaya yakın olan motorların yüzde kaçını yakaladık?

`accuracy_score`, `precision_score` ve `recall_score` fonksiyonlarını kullanın.

In [ ]:
# your code here

Uçak motorları için hangisi daha tehlikeli: **FP** mi, **FN** mi? Bu durumda hangi metrik daha önemli?

> **Cevabınızı buraya yazın (çift tıklayın)**

## 11. KPI'lar: sayıdan öneriye

Bir yönetici RMSE değerini sormaz; şunu sorar: **"Motorların kaçında tahmine güvenebilirim ve bakım uyarısı ne kadar işe yarıyor?"** Bu yüzden RMSE'yi iş diline çeviren KPI'lar hesaplayalım. Aşağıdaki kodu çalıştırın.

In [ ]:
hata = y_pred_rf - y_test_clip

print("Ortalama hata (MAE):", round(mae_rf, 1), "çevrim")
print("±10 çevrim içinde tahmin edilen motor: %", round((hata.abs() <= 10).mean() * 100))
print("±20 çevrim içinde tahmin edilen motor: %", round((hata.abs() <= 20).mean() * 100))

Şimdi **bakım uyarısı** kuralını deneyelim: Random Forest'in tahmini `esik` çevrimin altına düşerse alarm verilsin. Gerçekten arızaya yakın motorlar (`y_test_clip <= 30`) için:
- **Yakalanan arıza:** alarm verildi ve gerçekten arızaya yakın
- **Kaçırılan arıza:** alarm yok ama gerçekten arızaya yakın
- **Yanlış alarm:** alarm verildi ama motor aslında arızaya yakın değil

Eşiği 30, 40 ve 50 yaparak her seferinde bu üç sayıyı bulun ve `alarm_tablosu` adında bir tabloda toplayın.

<details>
    <summary> İpucu 💡 </summary>

`alarm = y_pred_rf <= esik` ve `gercek_yakin = y_test_clip <= 30` yazın. `alarm & gercek_yakin` yakalananı, `~alarm & gercek_yakin` kaçırılanı, `alarm & ~gercek_yakin` yanlış alarmı seçer. `.sum()` ile sayın.

</details>

In [ ]:
# your code here

Eşiği 30'dan 50'ye çıkarınca yakalanan arıza sayısı ve yanlış alarm nasıl değişti? Bakım ekibine hangi eşiği önerirsiniz ve neden?

> **Cevabınızı buraya yazın (çift tıklayın)**

## Sonuç

Bu çalışmada neler öğrendiniz? Bitirme raporunuz için en önemli 3 bulguyu yazın.

> **Cevabınızı buraya yazın (çift tıklayın)**